<a href="https://colab.research.google.com/github/LiaIssakov/Portfolio_2025/blob/main/dimensionality_reduction_with_tSNE_Fall26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dimensionality reduction and data visualization

The idea is to **visualize high-dimensional data** -- to the extent possible. To do this we will **reduce the dimension** of the data.

The idea is simple:
* Map each high-dimensional vector to a single vector in a low-dimensional space.

##First big question (that we won't answer today):
* **What properties or relationships within our high-dimensional data can we hope to retain in the low-dimensional view**?

* What **can't** we hope to preserve?





Original author: Hubert Wagner, UF 2021+

Improvements: Constantinos Barmpouris and Bruno Dede Jr, Fall 2023 students.

# Goals for today
* Get familiar with the concepts of dimensionality reduction and data visualization
* See how tSNE works on simple data (3D -> 2D)
* See how it works on our mnist dataset (optionally fashion mnist)

> Later we'll understand tSNE in much more detail.

## Our method of choice: tSNE

We will use tSNE (t-distributed stochastic neighbourhood embedding) to understand the structure of the mnist (digits) dataset some more.

It is an unsupervised learning method which will allow us to visualize the high-dimensional data in 2D (or 3D if we want).

> The good news is that -- for now! -- you don't have to understand any of the 4 complicated words above.

> Okay, 'embedding' means: 'choice of coordinates' in mathematics. We will 'embed' each high-dimensional vector by giving it two-dimensional coordinates. So it becomes a two-dimensional vector.


Note that with k-means we reduced the *number* of $d$-dimensional vectors (from n to k), but the dimensionality stayed the same. Here the result will be $n$ vectors in 2-dimensional space, so that we can plot them.

> Just like in the case of k-means -- we defer talking about algorithmic and mathematical details. For now we are happy with just using the method, since someone was nice enough to implement it.

# Technicalities

We use sklearn implementation of tSNE -- usage is very similar to k-means.

Today we will use:
- initialize model (setting some parameters)
- .fit function
- .embeddings_ (the result)
- We will normally use .fit_transform function instead of the two steps above. It fits and returns the embeddings in one go.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from keras.datasets import mnist, fashion_mnist
from sklearn.manifold import TSNE  # !

# technicalitiess below
plt.rcParams["figure.figsize"] = (15, 15) # larger default figures

# 3D $\to$ 2D intuition and technicalities

We'll generate some silly 3D data and then see how visualizing it in 2D works.

## Data generation (not important)

In [ ]:
def generate_3d_blob(n, thinness=5):
    '''No need to read the implementations!'''
    X3 = np.random.randn(n, 3)
    ns = np.linalg.norm(X3, axis=1, keepdims=True)
    X3 /= ns
    X3 *= (1 + np.random.randn(n, 1)/thinness)
    return X3


def generate_vaguely_interesting_3D_data(n):
    '''
    Returns roughly n points in 3D in a vaguely interesting configuration.

    No need to read the implementation!
    '''
    num_extra = int(np.sqrt(n))
    blobs = [generate_3d_blob(n//3) + np.random.randn(3)*5 for _ in range(3)]
    extra = []
    for b1 in blobs:
        for b2 in blobs:
            c1 = np.mean(b1, axis=0)
            c2 = np.mean(b2, axis=0)
            extra.append(np.linspace(c1, c2, num_extra)
                         + np.random.randn(num_extra, 3)/3)

    return np.concatenate(blobs + extra)

# tSNE on synthetic 3D data

In [ ]:
np.random.seed(5+1)
X3 = generate_vaguely_interesting_3D_data(1000)
ax = plt.axes(projection='3d')
ax.scatter3D(X3.T[0], X3.T[1], X3.T[2])  # we pass x,y,z now!
ax.axis('equal');

In [ ]:
tsne = TSNE(n_components=2)  # the 2 means we want to reduce the dimension to 2

X2 = tsne.fit_transform(X3)

# alternative, similar to how we used kMeans:
# tsne.fit(X3)
# X2 = tsne.embedding_

# now the i-th point of X2 is a 2d-vector representing the i-th vector in X3!
# so we can plot a 2d representation of our data
plt.scatter(X2.T[0], X2.T[1])

In [ ]:
X3.shape, X2.shape

# Important: tSNE is **not** a clustering method!

 You'll often see clusters, but (as you'll see later) it tends to overemphasise the existence of groups in the original dataset. In other words you'll often see disconnected groups of points -- this does not necessarily mean these were separate clusters in the original space!

# The Only Task

1. Fit tSNE with the **mnist dataset** (or fashion mnist if you prefer).

> Like kMeans, it requires a 2D array as input. Hopefully you know what to do.

> BTW: how many points can we reasonably handle? Let's start from only 2000.

2. Plot the resulting 2D embedding.
3. To verify the results scatter each resulting 2D points **colored by its correct label**.
4. Did it work?

> Note that tSNE is an unsupervised learning algorithm -- but **we use labels just to check** how well it performed. We don't use the labels for fitting the tNSE model.

In [ ]:
# note that we fit tSNE only with x_train
# but y_train will be useful for verification
(x_train, y_train), _ = mnist.load_data()
NUM_VECTORS = 2000  # PARAM
x_train = x_train[:NUM_VECTORS]/255.
y_train = y_train[:NUM_VECTORS]

In [ ]:
x_train.shape

In [ ]:
# Initialize the model

num_iters = 25  # PARAM <- controls speed vs accuracy
perplexity = 1  # PARAM <- larget means more neighbours are taken into account
# it's related to the number of neighbours considered, but not exactly this number
# we'll discuss it later!

my_random_state = 1236  # PARAM <- just makes sure we all see same stuff

# Init TSNE
tsne = TSNE(n_components=2, max_iter=num_iters,
            perplexity=perplexity, random_state=my_random_state)

# Check how the perplexity param changes the results

In [ ]:
# Fit the model -- similarly to k-means


In [ ]:
# Now get the new 'embedding' (coordinates of each input point).
# You can also do these two steps in one go, if you prefer
# In any case name it: x_reduced


In [ ]:
# Make an assert verifying the shape of resulting array?

In [ ]:
# assert x_reduced.shape == ...

## Plot the 2D embedding below

First just (scatter) plot the points.

In [ ]:
# just scatter the points

Add colors based on labels.

> Remember to color the points using the correct label of the original vector/image. So each reduced (2-dimensional) representations of a 0, should be plotted as a, say, green point etc.

> You can use the parameter 'c' of plt.scatter to supply an array of colors. It should be of the same length as the plotted points.

Note that tSNE is an unsupervised ML algorithm, but here we use labels to verify how well it worked, since we're learning to use it and don't really know if we should trust it.

In [ ]:
np.random.seed(0)

label_colors = np.random.random((10, 3))  # some random colors (RGB vectors!)
# now we have a colors baed on labels; it can be used with plt.scatter(..., c=?)
point_colors = label_colors[y_train]

# scatter the points with colors

# Useful: plotting tiny images in tSNE coordinates (it's all done, just run it)
... but it requires the previous part

> Also, this does not use the labels, so it's all unsupervised!

In [ ]:
# DON'T LOOK AT THE IMPLEMENTATION, it's just technicalities
from matplotlib.offsetbox import OffsetImage, AnnotationBbox


def plot_image_at_point(im, xy, zoom=1):
    '''
    Plots a tiny image at point xy.
    '''
    dxy = np.random.rand(2)/100 * plt.ylim()
    plt.arrow(*xy, *dxy)
    ab = AnnotationBbox(OffsetImage(im, zoom=zoom, cmap='gray_r'),
                        xy + dxy, frameon=False)
    plt.gca().add_artist(ab)

In [ ]:
np.random.seed(0)
plt.figure(figsize=(25, 25))  # for larger images!
image_zoom = 1.0  # controls the size of plotted inmages

plt.scatter(x_reduced.T[0], x_reduced.T[1]) # x and y coordinates
for im, xy in zip(x_train, x_reduced):
    plot_image_at_point(im, xy, image_zoom)

# Optional tasks:

If you're done, you can try to:
* test it on fashion mnist
* tune the PARAMs some more,
* Try embedding in $R^3$ (check the first example, plt.scatter doesn't work out of the box...)
* check what happens if you allow more/less data etc. What do you think the computational complexity is?